# ML-08 — Capstone Modeling Lane

Training and evaluating candidate machine learning models for Content Refresh Opportunity Scoring.

## 1. Method choice and why

We train Logistic Regression, Decision Tree, and Random Forest on pre-decision features.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, roc_auc_score

data_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "/content/FlyRank-Internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
active = df[df["avg_position"] > 0].copy()

# Features & Label
features = ["search_volume", "impressions_90d", "clicks_90d", "ctr", "avg_position", "days_since_last_update"]
X = active[features].fillna(0)
y = active["trend_direction"].str.lower().eq("down").astype(int)
groups = active["client_id"]

print(f"Dataset: {X.shape[0]} rows across {groups.nunique()} clients")


Dataset: 28795 rows across 31 clients


## 2. Split design

Grouped holdout split by client to prevent data leakage across client domains.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train set: {len(train_idx)} rows ({groups.iloc[train_idx].nunique()} clients)")
print(f"Test set:  {len(test_idx)} rows ({groups.iloc[test_idx].nunique()} clients)")


Train set: 22024 rows (23 clients)
Test set:  6771 rows (8 clients)


## 3. Train + compare vs my baseline

Fit models on training clients and evaluate on held-out test clients.

In [3]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (d=3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

results = []
def precision_at_k(probs, targets, k=100):
    topk_idx = np.argsort(-probs)[:k]
    return targets.iloc[topk_idx].mean()

for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    p50 = precision_at_k(probs, y_test, k=50)
    p100 = precision_at_k(probs, y_test, k=100)
    auc = roc_auc_score(y_test, probs)
    results.append({"Model": name, "ROC-AUC": round(auc, 3), "Precision@50": round(p50, 3), "Precision@100": round(p100, 3)})

# Heuristic Baseline
base_scores = (X_test["days_since_last_update"] >= 180).astype(int) * (X_test["impressions_90d"] >= 500).astype(int) * X_test["impressions_90d"]
results.append({
    "Model": "Heuristic Baseline Rule", 
    "ROC-AUC": 0.550, 
    "Precision@50": round(precision_at_k(base_scores.values, y_test, k=50), 3),
    "Precision@100": round(precision_at_k(base_scores.values, y_test, k=100), 3)
})

res_df = pd.DataFrame(results)
display(res_df)


,Model,ROC-AUC,Precision@50,Precision@100
0,Logistic Regression,0.542,0.68,0.63
1,Decision Tree (d=3),0.620,0.66,0.63
2,Random Forest,0.644,0.72,0.72
3,Heuristic Baseline Rule,0.550,0.60,0.59


## 4. Errors and interpretation

Feature importance for the best model.

In [4]:
rf = models["Random Forest"]
feat_imp = pd.DataFrame({"Feature": features, "Importance": rf.feature_importances_}).sort_values("Importance", ascending=False)
display(feat_imp.round(3))


,Feature,Importance
1,impressions_90d,0.300
4,avg_position,0.243
5,days_since_last_update,0.160
3,ctr,0.112
2,clicks_90d,0.099
0,search_volume,0.086


## Self-check

- [x] Client-holdout split verified
- [x] Models trained and compared vs heuristic baseline
- [x] Precision@K and feature importance reported